In [20]:
import os

In [21]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction'

In [4]:
os.chdir("..")

In [22]:
%pwd

'c:\\Users\\lenovo\\Desktop\\Kidney-Disease-Prediction'

In [31]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size:int
    params_is_augmentation: bool
    params_image_size: list

In [32]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [33]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_training_config(self) -> TrainingConfig:
        training= self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params
        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"Chest CT Scan data")
        create_directories([training.root_dir])
        
        training_config = TrainingConfig(
            root_dir = training.root_dir,
            trained_model_path = training.trained_model_path,
            updated_base_model_path = prepare_base_model.updated_base_model_path,
            training_data = training_data,
            params_epochs = params.EPOCHS,
            params_batch_size = params.BATCH_SIZE,
            params_is_augmentation = params.AUGMENTATION,
            params_image_size = params.IMAGE_SIZE
        )
        
        return training_config

In [34]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [35]:
class PrepareTrainingComponent:
    def __init__(self,config:TrainingConfig):
        self.config = config
        
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        
    def train_valid_model(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split = 0.20
        )
        
        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size = self.config.params_batch_size,
            # class_mode = "sparse"
            interpolation = "bilinear"
        )
        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )
        
        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
        
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator
            
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )
        
        
    @staticmethod
    def save_model(path:Path, model:tf.keras.Model):
        model.save(path)
        
    def train(self):
        self.steps_per_epoch = self.train_generator.samples  // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size
        
        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator,
            # callbacks=callback_list
        )
        
        self.save_model(
            self.config.trained_model_path,model=self.model
            )
        
# callback_list = [
#     ModelCheckpoint(filepath="best_model.h5", save_best_only=True),
#     EarlyStopping(patience=5)
# ]

        
    # def train_base_model(self,base_model):
    #     training_data = tf.keras.preprocessing.image_dataset_from_directory(
    #         self.config.training_data,
    #         image_size = self.config.params_image_size,
    #         batch_size = self.config.params_batch_size
    #     )
        
    #     base_model.compile(
    #         loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    #         optimizer = tf.keras.optimizers.Adam(),
    #         metrics = ['accuracy']
    #     )
        
    #     history = base_model.fit(
    #         training_data,
    #         epochs = self.config.params_epochs
    #     )
        
    #     base_model.save(self.config.trained_model_path)
        
    #     return history

In [ ]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = PrepareTrainingComponent(config=training_config)
    training.get_base_model()
    training.train_valid_model()
    training.train() 
except Exception as e:
    raise e

[2026-01-30 22:16:15,352: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-01-30 22:16:15,360: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-30 22:16:15,363: INFO: common: created directory at: artifacts]
[2026-01-30 22:16:15,366: INFO: common: created directory at: artifacts/training]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
261/368 [====================>.........] - ETA: 14:47 - loss: 10.4962 - accuracy: 0.6231